# Save simulation experiments

Each execution creates a **new subfolder inside `bilinear_regression/results`**, labelled by experiment name, a fingerprint of its true betas, and a timestamp. Existing results are never overwritten. This notebook fits simulated outcomes, not age.

Edit the ROI blocks below. Indices are **0-based** and the stop index is excluded: `(40, 60, 1.5)` assigns 1.5 to ROIs 40–59. Duplicate the experiment dictionary to run multiple different true-beta configurations.

CV uses `parameters.json`. The final model defaults to the lambda pair with minimum mean CV validation MSE; true betas are used only for data generation and coefficient comparison. Saving connectivity makes each run self-contained but may use hundreds of MB. To regenerate plots later, open **`plot_saved_simulation.ipynb`**, which does not refit anything.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_DIR = Path.cwd() / "model"
if not (MODEL_DIR / "simulation_results.py").exists():
    MODEL_DIR = Path.cwd()
sys.path.insert(0, str(MODEL_DIR))
from simulation_results import (
    RESULTS_DIR, betas_from_blocks, run_simulation_experiment,
    load_simulation_experiment, plot_saved_betas,
)
from config import PARAMS


In [ ]:
EXPERIMENTS = [
    {
        "name": "original_blocks",
        "beta1_blocks": [(40, 60, 1.5), (120, 150, -1.0)],
        "beta2_blocks": [(40, 60, 1.0), (120, 150, -1.2)],
    },
    # Duplicate the dictionary above, change its name and blocks, and add it here.
]

# Copies parameters.json; overrides here affect these runs only.
RUN_PARAMETERS = dict(PARAMS)
# Example for a shorter exploratory run:
# RUN_PARAMETERS.update(LAMBDA1_GRID=[1, 2], LAMBDA2_GRID=[1, 2])
SEED = 2026
SAVE_CONNECTIVITY = True
FINAL_LAMBDA_PAIR = None  # Or specify (1.0, 1.0) to keep fixed penalties.
print(f"{len(EXPERIMENTS)} experiment(s); "
      f"{len(RUN_PARAMETERS['LAMBDA1_GRID']) * len(RUN_PARAMETERS['LAMBDA2_GRID'])} "
      f"lambda pairs x {RUN_PARAMETERS['N_SPLITS']} folds each")

In [ ]:
saved_runs = []
for experiment in EXPERIMENTS:
    true1, true2 = betas_from_blocks(
        RUN_PARAMETERS["P"], experiment["beta1_blocks"], experiment["beta2_blocks"]
    )
    run_dir = run_simulation_experiment(
        experiment["name"], true1, true2,
        parameters=RUN_PARAMETERS, seed=SEED,
        block_definitions=experiment,
        final_lambda_pair=FINAL_LAMBDA_PAIR,
        save_connectivity=SAVE_CONNECTIVITY,
    )
    saved_runs.append(run_dir)
    print(f"Completed: {run_dir}")

In [ ]:
# These plots use the saved files, not an in-memory fitted model.
for run_dir in saved_runs:
    saved = load_simulation_experiment(run_dir)
    display(saved["cv_summary"])
    plot_saved_betas(saved, run_dir / "figures" / "true_vs_estimated_betas.png")
    plt.show()